# ARDAP - Modelo de priorizacao de adocao

**Checkpoint 2** - componente de IA funcionando com as primeiras metricas.

**Problema:** a ONG resgata mais animais do que consegue divulgar. O modelo estima
a probabilidade de um animal ser adotado em ate 30 dias para apoiar a decisao de
quem divulgar primeiro.

**Eixo:** Machine Learning (classificacao binaria).

**Dados:** Austin Animal Center (dados abertos, dominio publico), via API Socrata.
O dicionario e a licenca estao em `datasets/README.md`.

> Este notebook reaproveita as funcoes de `ai-model/scripts/treinar.py` para nao
> duplicar codigo. O script e a fonte da verdade; aqui a gente acompanha e
> interpreta cada etapa.

In [ ]:
from pathlib import Path
import sys

# Sobe ate a raiz do repositorio (pasta que contem ai-model/)
RAIZ = Path.cwd()
while not (RAIZ / "ai-model").exists():
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "ai-model"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scripts.treinar import (
    SEMENTE,
    JANELA_ADOCAO,
    carregar_dados,
    montar_base,
    preparar_features,
    construir_modelos,
    construir_preprocessamento,
    avaliar,
    melhor_threshold,
)

pd.set_option("display.width", 120)
print("raiz:", RAIZ)
print("semente:", SEMENTE, "| janela:", JANELA_ADOCAO, "dias")

## 1. Dados

Se os arquivos ainda nao existirem, a celula abaixo baixa (~30 MB, fora do Git).

In [ ]:
if not (RAIZ / "datasets" / "raw" / "intakes.csv").exists():
    from scripts.baixar_dados import baixar
    baixar("intakes", RAIZ / "datasets" / "raw" / "intakes.csv")
    baixar("outcomes", RAIZ / "datasets" / "raw" / "outcomes.csv")

intakes, outcomes = carregar_dados(RAIZ / "datasets" / "raw")
print(f"entradas: {len(intakes):,} | desfechos: {len(outcomes):,}")
intakes.head(3)

## 2. Alvo

Cada linha e uma **entrada** de animal no abrigo. O alvo compara a data de entrada
com o primeiro desfecho posterior:

- `1` se o desfecho foi `Adoption` em ate 30 dias;
- `0` se teve outro desfecho, adocao tardia ou continuou no abrigo depois de 30 dias;
- linhas sem desfecho e ainda dentro da janela sao **censuradas** e descartadas.

In [ ]:
base = montar_base(intakes, outcomes)
print(f"registros usaveis: {len(base):,}")
print(f"taxa de positivos: {base['y'].mean():.1%}")
print(f"periodo: {base['datetime'].min():%m/%Y} a {base['datetime'].max():%m/%Y}")

base["y"].value_counts().rename({0: "nao adotado em 30d", 1: "adotado em 30d"})

### Perfil dos dados: taxa de adocao por especie

In [ ]:
resumo = (
    base.groupby("animal_type")["y"]
    .agg(taxa="mean", animais="size")
    .query("animais >= 200")
    .sort_values("animais", ascending=False)
)
resumo["taxa"] = (resumo["taxa"] * 100).round(1)
resumo

## 3. Features

Usamos apenas campos que o cadastro do ARDAP tambem tem (ou tera):

| Austin | ARDAP |
|---|---|
| `animal_type` | `especie` |
| `breed` | `raca` |
| `sex_upon_intake` | `sexo` + `castrado` |
| `age_upon_intake` | `idade_estimada` |
| `color` | `cor_pelo` |
| `intake_type` | `origem_resgate` |
| `intake_condition` | situacao de saude do resgate |

`porte`, `peso` e `temperamento` entram quando o modelo for re-treinado com dados
proprios da ONG.

In [ ]:
X = preparar_features(base)
y = base["y"].astype(int)
print(X.dtypes)
X.head(3)

## 4. Treino e validacao

Split estratificado 70/15/15. Selecao do modelo na validacao por F1 macro.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

X_tr, X_resto, y_tr, y_resto = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEMENTE)
X_val, X_te, y_val, y_te = train_test_split(X_resto, y_resto, test_size=0.50, stratify=y_resto, random_state=SEMENTE)
print(f"treino {len(X_tr):,} | validacao {len(X_val):,} | teste {len(X_te):,}")

modelos = {}
resultados = {}
for nome, estimador in construir_modelos().items():
    modelo = Pipeline([("preparo", construir_preprocessamento()), ("modelo", estimador)])
    modelo.fit(X_tr, y_tr)
    modelos[nome] = modelo
    resultados[nome] = avaliar(modelo, X_val, y_val)

tabela = pd.DataFrame(resultados).T[["f1_macro", "f1", "roc_auc", "pr_auc", "precisao", "recall"]]
tabela.round(3).sort_values("f1_macro", ascending=False)

## 5. Avaliacao no teste

O melhor modelo da validacao e o threshold que maximiza F1 sao aplicados **uma vez**
no conjunto de teste. Comparar com o baseline mostra se o modelo aprendeu algo alem
da classe majoritaria.

In [ ]:
melhor_nome = max(resultados, key=lambda n: resultados[n]["f1_macro"])
melhor = modelos[melhor_nome]
threshold, _ = melhor_threshold(melhor, X_val, y_val)
print(f"melhor modelo: {melhor_nome} | threshold: {threshold:.2f}")

from sklearn.metrics import classification_report

proba = melhor.predict_proba(X_te)[:, 1]
pred = (proba >= threshold).astype(int)
print(classification_report(y_te, pred, target_names=["nao adotado", "adotado"], digits=3, zero_division=0))

pd.DataFrame({n: avaliar(m, X_te, y_te) for n, m in modelos.items()}).T[
    ["f1_macro", "f1", "roc_auc", "pr_auc", "precisao", "recall", "acuracia"]
].round(3)

### Matriz de confusao

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(4.5, 4))
ConfusionMatrixDisplay.from_predictions(y_te, pred, display_labels=["nao", "sim"], ax=ax, colorbar=False, cmap="Greens")
ax.set_title(f"Matriz de confusao - {melhor_nome}")
plt.show()

## 6. Interpretacao

Permutation importance mede quanto o ROC-AUC cai quando embaralhamos uma coluna.
E a base dos "fatores" que a tela vai mostrar para o funcionario.

In [ ]:
from sklearn.inspection import permutation_importance

amostra = X_te.sample(n=min(4000, len(X_te)), random_state=SEMENTE)
resultado = permutation_importance(
    melhor, amostra, y_te.loc[amostra.index],
    n_repeats=5, random_state=SEMENTE, scoring="roc_auc", n_jobs=-1,
)
importancias = (
    pd.DataFrame({"feature": amostra.columns, "importancia": resultado.importances_mean})
    .sort_values("importancia", ascending=False)
    .reset_index(drop=True)
)
importancias.head(10)

In [ ]:
top = importancias.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(top["feature"], top["importancia"], color="#ef6c00")
ax.set_xlabel("queda media no ROC-AUC")
ax.set_title("Fatores mais relevantes")
plt.show()

## 7. Conclusao

Preencher depois da execucao com os numeros reais:

- O modelo supera o baseline em F1 macro? Quanto?
- Onde ele erra mais (falsos positivos ou falsos negativos)? Qual erro e pior para a ONG?
- Quais fatores pesaram mais e isso faz sentido no dominio?

**Limitacoes:** base dos EUA, alvo de 30 dias, censura a direita. O modelo apoia a
decisao de divulgacao e nao deve ser usado para negar cuidado a nenhum animal.

In [ ]:
# Salva o modelo no formato usado pelo backend (joblib)
import joblib

destino = RAIZ / "ai-model" / "models" / "modelo.joblib"
destino.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(
    {
        "pipeline": melhor,
        "threshold": threshold,
        "features": list(X.columns),
        "janela_dias": JANELA_ADOCAO,
        "semente": SEMENTE,
    },
    destino,
)
print("modelo salvo em", destino)